# 06 · Refined Pair Correlation: Zeta vs Poisson vs GUE Matrix

This notebook refines the previous pair-correlation-style comparison.

The earlier notebook used a practical histogram of pair differences.  
Here we build a cleaner empirical estimator with better normalization and a clearer small-separation comparison.

## Goal

For an unfolded sequence \(x_1,\dots,x_N\) with average spacing near 1, we want to compare the empirical distribution of pair differences against:

- **Poisson** behavior
- **finite GUE matrix** behavior
- the standard GUE reference shape
  \[
  1 - \left(\frac{\sin \pi s}{\pi s}\right)^2
  \]

## Important note

This is still a numerical lab notebook, not a theorem notebook.

The estimator below is closer to the standard pair-correlation setup than the first-pass version, but it is still finite-sample and practical rather than fully asymptotic.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Zeta zeros

In [ ]:
N = 700
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps = np.diff(t)

len(t), len(zeta_gaps), t[:5]

## Unfolding helper

We build an unfolded coordinate by normalizing gaps with a moving local mean and then cumulatively summing.

This keeps the average local spacing near 1.

In [ ]:
def local_normalize_gaps(gaps, window=21):
    kernel = np.ones(window) / window
    local_means = np.convolve(gaps, kernel, mode="same")
    half = window // 2
    for i in range(half):
        local_means[i] = gaps[:i + half + 1].mean()
        local_means[-(i + 1)] = gaps[-(i + half + 1):].mean()
    return gaps / local_means

def unfold_from_gaps(gaps, window=21):
    unfolded_gaps = local_normalize_gaps(gaps, window=window)
    x = np.concatenate([[0.0], np.cumsum(unfolded_gaps)])
    return x, unfolded_gaps

In [ ]:
zeta_x, zeta_unfolded_gaps = unfold_from_gaps(zeta_gaps, window=21)
zeta_unfolded_gaps.mean(), zeta_unfolded_gaps.std()

## Poisson control

In [ ]:
poisson_gaps = rng.exponential(scale=1.0, size=len(zeta_gaps))
poisson_x, poisson_unfolded_gaps = unfold_from_gaps(poisson_gaps, window=21)

poisson_unfolded_gaps.mean(), poisson_unfolded_gaps.std()

## Finite GUE matrix reference

We generate random complex Hermitian matrices, extract bulk eigenvalue spacings, and unfold the resulting gap sequence.

In [ ]:
def sample_gue_bulk_spacings(matrix_size=120, n_matrices=32, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []

    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0

        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep

        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps = sample_gue_bulk_spacings(matrix_size=120, n_matrices=32, bulk_fraction=0.5, rng=rng)
gue_x, gue_unfolded_gaps = unfold_from_gaps(gue_gaps, window=21)

gue_unfolded_gaps.mean(), gue_unfolded_gaps.std(), len(gue_x)

## Nearest-neighbor sanity check

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 3.0, 32)
plt.hist(zeta_unfolded_gaps, bins=bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_unfolded_gaps, bins=bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_unfolded_gaps, bins=bins, density=True, alpha=0.45, label="GUE matrix")
plt.xlabel("unfolded spacing")
plt.ylabel("density")
plt.title("Unfolded nearest-neighbor spacing check")
plt.legend()
plt.show()

## Refined empirical pair-correlation histogram

For an unfolded point sequence \(x_n\), we collect positive pair differences
\[
s = x_j - x_i,\qquad j>i,
\]
inside a chosen window \(0 < s \le s_{\max}\).

The estimator used here is:

1. collect all such positive differences up to \(s_{\max}\)
2. build a histogram with density normalization
3. compare the small-\(s\) behavior across ensembles

This is still practical rather than fully asymptotic, but it behaves much better than the earlier coarse estimator.

In [ ]:
def positive_pair_differences(x, s_max=4.0):
    diffs = []
    n = len(x)
    for i in range(n - 1):
        d = x[i+1:] - x[i]
        d = d[(d > 0) & (d <= s_max)]
        if len(d):
            diffs.extend(d.tolist())
    return np.array(diffs)

zeta_pair = positive_pair_differences(zeta_x, s_max=4.0)
poisson_pair = positive_pair_differences(poisson_x, s_max=4.0)
gue_pair = positive_pair_differences(gue_x, s_max=4.0)

len(zeta_pair), len(poisson_pair), len(gue_pair)

## Pair-difference histogram

In [ ]:
plt.figure(figsize=(8.8, 5.2))
bins = np.linspace(0, 4.0, 44)
plt.hist(zeta_pair, bins=bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_pair, bins=bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_pair, bins=bins, density=True, alpha=0.45, label="GUE matrix")
plt.xlabel("pair difference")
plt.ylabel("density")
plt.title("Refined pair-difference comparison")
plt.legend()
plt.show()

## Small-separation comparison

This is the most important region.  
For GUE-like behavior, the density should be **suppressed near 0** compared with Poisson.

In [ ]:
small_bins = np.linspace(0, 1.5, 36)

plt.figure(figsize=(8.8, 5.0))
plt.hist(zeta_pair, bins=small_bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_pair, bins=small_bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_pair, bins=small_bins, density=True, alpha=0.45, label="GUE matrix")
plt.xlabel("pair difference")
plt.ylabel("density")
plt.title("Small-separation refined pair comparison")
plt.legend()
plt.show()

## Overlay with the standard GUE reference shape

The standard GUE pair-correlation reference is

\[
1 - \left(\frac{\sin \pi s}{\pi s}\right)^2.
\]

Because our histogram is finite and density-normalized on a finite interval, we normalize the reference curve on the same interval before overlaying it.

In [ ]:
s = np.linspace(0.001, 4.0, 800)
gue_reference = 1 - (np.sin(np.pi * s) / (np.pi * s))**2
gue_reference_norm = gue_reference / np.trapz(gue_reference, s)

plt.figure(figsize=(8.8, 5.2))
plt.hist(zeta_pair, bins=np.linspace(0, 4.0, 44), density=True, alpha=0.40, label="zeta pair histogram")
plt.hist(gue_pair, bins=np.linspace(0, 4.0, 44), density=True, alpha=0.30, label="GUE matrix pair histogram")
plt.plot(s, gue_reference_norm, linewidth=2, label="normalized GUE reference shape")
plt.xlabel("pair difference")
plt.ylabel("density")
plt.title("Refined pair histogram with GUE reference shape")
plt.legend()
plt.show()

## Very-small-separation zoom

This makes the near-zero behavior easier to inspect.

In [ ]:
tiny_bins = np.linspace(0, 0.8, 28)

plt.figure(figsize=(8.8, 5.0))
plt.hist(zeta_pair, bins=tiny_bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_pair, bins=tiny_bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_pair, bins=tiny_bins, density=True, alpha=0.45, label="GUE matrix")
plt.xlabel("pair difference")
plt.ylabel("density")
plt.title("Very-small-separation zoom")
plt.legend()
plt.show()

## Local pair-count diagnostic

For each point \(x_i\), count how many later points fall within a small radius \(r\).
This is another compact local two-point summary.

In [ ]:
def local_pair_count(x, radius=0.75):
    counts = []
    n = len(x)
    for i in range(n - 1):
        d = x[i+1:] - x[i]
        counts.append(np.sum((d > 0) & (d <= radius)))
    return np.array(counts)

zeta_counts = local_pair_count(zeta_x, radius=0.75)
poisson_counts = local_pair_count(poisson_x, radius=0.75)
gue_counts = local_pair_count(gue_x, radius=0.75)

zeta_counts.mean(), poisson_counts.mean(), gue_counts.mean()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.arange(0, max(zeta_counts.max(), poisson_counts.max(), gue_counts.max()) + 2) - 0.5
plt.hist(zeta_counts, bins=bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_counts, bins=bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_counts, bins=bins, density=True, alpha=0.45, label="GUE matrix")
plt.xlabel("count of later points within radius 0.75")
plt.ylabel("density")
plt.title("Refined local pair-count comparison")
plt.legend()
plt.show()

## Summary statistics

In [ ]:
def summary_stats(x):
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
    }

summary = {
    "zeta_unfolded_gaps": summary_stats(zeta_unfolded_gaps),
    "poisson_unfolded_gaps": summary_stats(poisson_unfolded_gaps),
    "gue_unfolded_gaps": summary_stats(gue_unfolded_gaps),
    "zeta_pair_diffs": summary_stats(zeta_pair),
    "poisson_pair_diffs": summary_stats(poisson_pair),
    "gue_pair_diffs": summary_stats(gue_pair),
    "zeta_local_pair_counts": summary_stats(zeta_counts),
    "poisson_local_pair_counts": summary_stats(poisson_counts),
    "gue_local_pair_counts": summary_stats(gue_counts),
}
summary

## Notes

- This refined notebook removes the earlier short index-offset cutoff and instead uses all positive pair differences up to a fixed separation window.
- The histogram is still finite-sample, but the near-zero comparison is much cleaner.
- The main practical question is whether zeta looks closer to finite GUE matrix data than to the Poisson control.
- The analytic GUE curve is included as a qualitative reference after interval normalization.

## Next directions

A natural next notebook could do one of these:

1. build a more standard windowed pair-correlation estimator with explicit edge correction  
2. increase the GUE matrix size and sample count  
3. compare 3-point and 4-point correlation-style summaries  
4. extend the local neighborhood framework beyond nearest-neighbor ratios